# Assignment: Build It, Then Break It — Mini Transformer

**Name:** [Your Name]  
**Student ID:** [Your Student ID]  
**Course:** NLP Lab  
**Date:** November 24, 2026

## 1. Introduction

> *[Write 2–3 sentences in your own words explaining what you are building and what you are testing in this lab.]*

## 2. Baseline Model

In Part A, we build and train a character-level, decoder-only Mini Transformer (`MiniGPT`) on the **Tiny Shakespeare** dataset (~1.1 MB, 90% train / 10% validation split) using the required configuration:

| Setting | Value |
| :--- | :--- |
| Tokenization | Character-level |
| Context length (`block_size`) | 128 |
| Embedding size (`d_model`) | 128 |
| Number of layers (`n_layer`) | 4 |
| Attention heads (`n_head`) | 4 |
| Feed-forward hidden size | $4 \times d_{\text{model}} = 512$ |
| Positional encoding | Sinusoidal (toggleable via config flag) |
| Dropout | 0.1 |
| Optimizer | AdamW (`lr = 3e-4`) |
| Batch size | 64 |
| Training steps | 1,500 |

In [ ]:
# Step 0: Imports, GPU check, and reproducibility helper
import math
import os
import json
import random
import time
import urllib.request
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 42

def set_seed(seed: int = SEED):
    """Fix random seeds across Python, NumPy, and PyTorch for controlled experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"PyTorch version: {torch.__version__}")
print(f"Using device:    {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. In Google Colab, switch to GPU via Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# Step 1: Download Tiny Shakespeare, build vocabulary, encode/decode, and 90/10 train-val split
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"

if not os.path.exists(DATA_PATH):
    print(f"Downloading Tiny Shakespeare from {DATA_URL} ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete!")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters (~{os.path.getsize(DATA_PATH) / 1e6:.2f} MB)")
print("First 200 characters:\n" + "-" * 40)
print(text[:200])
print("-" * 40)

# Build character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    """Convert a string into a list of integer token IDs."""
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    """Convert a list of integer token IDs back into a string."""
    return "".join([itos[i] for i in ids])

print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Vocab preview:   {repr(''.join(chars))}")

# Quick round-trip check
sample_str = "First Citizen:"
assert decode(encode(sample_str)) == sample_str, "Encode/decode round-trip failed!"

# Encode full dataset and split 90% train / 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data):,} | Validation tokens: {len(val_data):,}")

In [ ]:
# Step 2: Data batching helper
def get_batch(split: str, block_size: int, batch_size: int):
    """Sample a random batch of input chunks (x) and next-character targets (y)."""
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# Verify shapes and 1-character shift
xb, yb = get_batch("train", block_size=128, batch_size=64)
print(f"Batch shapes -> x: {tuple(xb.shape)}, y: {tuple(yb.shape)}")
print(f"Example input  (first 25 chars): {repr(decode(xb[0, :25].tolist()))}")
print(f"Example target (first 25 chars): {repr(decode(yb[0, :25].tolist()))}")

In [ ]:
# Steps 3 & 4: Config flags and modular MiniGPT classes
@dataclass
class GPTConfig:
    # Core architecture settings (Part A defaults)
    vocab_size: int = vocab_size
    block_size: int = 128
    d_model: int = 128
    n_layer: int = 4
    n_head: int = 4
    dropout: float = 0.1
    # Training settings
    batch_size: int = 64
    lr: float = 3e-4
    max_iters: int = 1500
    eval_interval: int = 125   # Evaluate smoothly every 125 steps
    print_interval: int = 250  # Print progress every 250 steps (per assignment)
    eval_iters: int = 40
    seed: int = SEED
    # Ablation flags (Part B)
    use_causal_mask: bool = True   # B1: set False to remove causal mask
    use_pos_encoding: bool = True  # B2: set False to remove sinusoidal positional encoding
    use_scale: bool = True         # B3: set False to remove 1/sqrt(d_k) scaling
    use_residual: bool = True      # B5: set False to remove residual connections


class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encodings from 'Attention Is All You Need' (Vaswani et al., 2017)."""
    def __init__(self, block_size: int, d_model: int):
        super().__init__()
        pe = torch.zeros(block_size, d_model)
        position = torch.arange(0, block_size, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)  # shape: (block_size, d_model)

    def forward(self, seq_len: int) -> torch.Tensor:
        return self.pe[:seq_len, :]  # shape: (T, d_model)


class Head(nn.Module):
    """One self-attention head (causal & scaled by default, configurable for ablations)."""
    def __init__(self, config: GPTConfig, head_size: int):
        super().__init__()
        self.head_size = head_size
        self.use_causal_mask = config.use_causal_mask
        self.use_scale = config.use_scale

        self.key = nn.Linear(config.d_model, head_size, bias=False)
        self.query = nn.Linear(config.d_model, head_size, bias=False)
        self.value = nn.Linear(config.d_model, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(config.block_size, config.block_size)))
        self.dropout = nn.Dropout(config.dropout)
        self.last_attn_weights = None  # Used only when measuring attention entropy (B3)

    def forward(self, x: torch.Tensor, capture_attn: bool = False) -> torch.Tensor:
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Scaled (or unscaled for B3) dot-product attention scores
        wei = q @ k.transpose(-2, -1)  # (B, T, T)
        if self.use_scale:
            wei = wei * (self.head_size ** -0.5)

        # Causal mask (prevent attending to future positions; removed in B1)
        if self.use_causal_mask:
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        attn_probs = F.softmax(wei, dim=-1)  # (B, T, T)
        if capture_attn:
            self.last_attn_weights = attn_probs.detach()
        else:
            self.last_attn_weights = None

        wei = self.dropout(attn_probs)
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v      # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """Multiple self-attention heads running in parallel, concatenated and projected."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.d_model % config.n_head == 0, "d_model must be divisible by n_head"
        head_size = config.d_model // config.n_head
        self.heads = nn.ModuleList([Head(config, head_size) for _ in range(config.n_head)])
        self.proj = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor, capture_attn: bool = False) -> torch.Tensor:
        out = torch.cat([h(x, capture_attn=capture_attn) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """Position-wise feed-forward network (d_model -> 4*d_model -> d_model)."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Pre-LayerNorm Transformer block: Attention + FeedForward with optional residual connections."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.use_residual = config.use_residual
        self.sa = MultiHeadAttention(config)
        self.ffwd = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.d_model)
        self.ln2 = nn.LayerNorm(config.d_model)

    def forward(self, x: torch.Tensor, capture_attn: bool = False) -> torch.Tensor:
        if self.use_residual:
            x = x + self.sa(self.ln1(x), capture_attn=capture_attn)
            x = x + self.ffwd(self.ln2(x))
        else:
            x = self.sa(self.ln1(x), capture_attn=capture_attn)
            x = self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    """Character-level decoder-only Transformer language model."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(config.block_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None, capture_layer1_attn: bool = False):
        B, T = idx.shape
        x = self.token_embedding_table(idx)  # (B, T, d_model)
        if self.config.use_pos_encoding:
            x = x + self.pos_encoding(T)     # (T, d_model) broadcasts over batch B
        x = self.drop(x)

        for i, block in enumerate(self.blocks):
            x = block(x, capture_attn=(capture_layer1_attn and i == 0))

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int = 500, temperature: float = 1.0) -> torch.Tensor:
        """Autoregressively sample new tokens one character at a time."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx

    @torch.no_grad()
    def compute_layer1_attention_entropy(self, idx: torch.Tensor) -> float:
        """Compute mean Shannon entropy (in nats) of Layer 1 attention weights across batches, heads, and positions."""
        self.eval()
        _ = self(idx, capture_layer1_attn=True)
        entropies = []
        eps = 1e-12
        for head in self.blocks[0].sa.heads:
            attn = head.last_attn_weights  # (B, T, T)
            ent = -(attn * torch.log(attn + eps)).sum(dim=-1)  # (B, T)
            entropies.append(ent.mean().item())
            head.last_attn_weights = None
        self.train()
        return float(np.mean(entropies))

print("Model classes defined successfully.")

In [ ]:
# Step 5: Reusable training, evaluation, and plotting utilities
@torch.no_grad()
def estimate_metrics(model: MiniGPT, config: GPTConfig, measure_entropy: bool = False):
    """Estimate train loss, validation loss, and (optionally) Layer 1 attention entropy."""
    model.eval()
    metrics = {}
    for split in ["train", "val"]:
        losses = torch.zeros(config.eval_iters)
        entropies = []
        for k in range(config.eval_iters):
            X, Y = get_batch(split, config.block_size, config.batch_size)
            _, loss = model(X, Y)
            losses[k] = loss.item()
            if measure_entropy and split == "val" and k < 5:
                entropies.append(model.compute_layer1_attention_entropy(X))
        metrics[f"{split}_loss"] = losses.mean().item()
        if measure_entropy and split == "val":
            metrics["layer1_entropy"] = float(np.mean(entropies))
    model.train()
    return metrics


def run_experiment(name: str, config: GPTConfig, generate_chars: int = 500, measure_entropy: bool = True):
    """Train a MiniGPT model from scratch with a fixed seed and return its metrics and sample."""
    set_seed(config.seed)
    model = MiniGPT(config).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)

    print(f"=== Running: {name} ===")
    print(f"Parameters: {n_params:,} | n_layer={config.n_layer}, n_head={config.n_head}, "
          f"causal={config.use_causal_mask}, pos_enc={config.use_pos_encoding}, "
          f"scale={config.use_scale}, residual={config.use_residual}")

    history = {"steps": [], "train_loss": [], "val_loss": [], "layer1_entropy": []}
    start_time = time.time()

    for step in range(config.max_iters + 1):
        # Periodic evaluation
        if step % config.eval_interval == 0 or step == config.max_iters:
            m = estimate_metrics(model, config, measure_entropy=measure_entropy)
            history["steps"].append(step)
            history["train_loss"].append(m["train_loss"])
            history["val_loss"].append(m["val_loss"])
            if measure_entropy:
                history["layer1_entropy"].append(m["layer1_entropy"])

            if step % config.print_interval == 0 or step == config.max_iters:
                ent_str = f" | L1 attn entropy: {m['layer1_entropy']:.4f}" if measure_entropy else ""
                print(f"Step {step:4d}/{config.max_iters} | "
                      f"train loss: {m['train_loss']:.4f} | val loss: {m['val_loss']:.4f}{ent_str}")

        if step == config.max_iters:
            break

        # Standard training step
        xb, yb = get_batch("train", config.block_size, config.batch_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    elapsed = time.time() - start_time
    print(f"Finished {name} in {elapsed:.1f}s | "
          f"Final Train Loss: {history['train_loss'][-1]:.4f} | "
          f"Final Val Loss: {history['val_loss'][-1]:.4f}")

    # Generate 500-character sample seeded with a newline character
    sample_text = ""
    if generate_chars > 0:
        set_seed(config.seed)
        start_token = stoi.get("\n", 0)
        context = torch.tensor([[start_token]], dtype=torch.long, device=DEVICE)
        generated_ids = model.generate(context, max_new_tokens=generate_chars)[0].tolist()
        sample_text = decode(generated_ids[1:])  # exclude seed newline

    result = {
        "name": name,
        "config": asdict(config),
        "n_params": n_params,
        "elapsed_sec": round(elapsed, 2),
        "history": history,
        "final_train_loss": history["train_loss"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_layer1_entropy": history["layer1_entropy"][-1] if measure_entropy else None,
        "sample": sample_text,
    }
    return model, result


def plot_loss_comparison(runs: list[dict], title: str, show_train: bool = True, show_val: bool = True):
    """Plot multiple runs on the exact same axes for fair comparison."""
    plt.figure(figsize=(9, 5))
    colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e"]

    for idx, run in enumerate(runs):
        c = colors[idx % len(colors)]
        steps = run["history"]["steps"]
        label_base = run["name"]
        if show_train:
            plt.plot(steps, run["history"]["train_loss"], linestyle="--", color=c, alpha=0.65,
                     label=f"{label_base} — Train")
        if show_val:
            plt.plot(steps, run["history"]["val_loss"], linestyle="-", marker="o", markersize=4,
                     linewidth=2, color=c, label=f"{label_base} — Val")

    plt.title(title, fontsize=13, fontweight="bold")
    plt.xlabel("Training Steps", fontsize=11)
    plt.ylabel("Cross-Entropy Loss", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

# Store all experiment outputs here for easy summary & saving
results = {}
print("Training and plotting utilities ready.")

In [ ]:
# Step 6: Train Baseline MiniGPT (Part A)
baseline_config = GPTConfig()
baseline_model, results["Baseline"] = run_experiment(
    name="Baseline (4L, 4H)",
    config=baseline_config,
    generate_chars=500,
    measure_entropy=True,
)

In [ ]:
# Plot Baseline train & validation loss curves and inspect the 500-character generated sample
plot_loss_comparison([results["Baseline"]], title="Part A: Baseline MiniGPT — Train vs. Validation Loss")

print("=" * 60)
print("BASELINE — 500-CHARACTER GENERATED SAMPLE")
print("=" * 60)
print(results["Baseline"]["sample"])
print("=" * 60)

### Part A Analysis (Your Own Words)

> *[Write one paragraph in your own words describing what the baseline model has and has not learned — comment specifically on spelling, real vs. invented words, formatting/line breaks/speaker labels, and overall meaning/coherence.]*

## 3. Ablation Experiments

In Part B, we change **one component at a time**, keeping every other hyperparameter and the random seed (`SEED = 42`) identical to the Baseline.  
Remember the **Prediction Rule**: write your prediction in the markdown cell *before* running the experiment cell, and do not edit it afterwards.

### B1: Removing the Causal Mask

**Prediction (write before running B1 — do not edit afterwards):**

- **Experiment:** B1 — Remove the causal mask (`use_causal_mask = False`)
- **I predict:** *[Write what you expect to happen to train loss, validation loss, and the generated text.]*
- **Because:** *[Explain your reasoning based on how causal self-attention works.]*

In [ ]:
# Run Experiment B1: Remove the causal mask
b1_config = GPTConfig(use_causal_mask=False)
b1_model, results["B1"] = run_experiment(
    name="B1: No Causal Mask",
    config=b1_config,
    generate_chars=500,
    measure_entropy=True,
)

# Compare loss curves on the same plot and axes
plot_loss_comparison(
    [results["Baseline"], results["B1"]],
    title="Experiment B1: Baseline vs. No Causal Mask",
)

print("=" * 60)
print("B1 (NO CAUSAL MASK) — 500-CHARACTER GENERATED SAMPLE")
print("=" * 60)
print(results["B1"]["sample"])
print("=" * 60)

#### B1 Report (Your Own Words)

- **Result (with numbers):** *[State the final train loss, final validation loss, and sample quality of B1 compared to Baseline in 1–2 sentences.]*
- **Explanation (and comparison to prediction):** *[Answer: Why does the loss look much better while the generated text gets worse? Explain where your prediction was right or wrong.]*

### B2: Removing Positional Encoding

**Prediction (write before running B2 — do not edit afterwards):**

- **Experiment:** B2 — Remove sinusoidal positional encoding (`use_pos_encoding = False`)
- **I predict:** *[Write what you expect to happen to train loss, validation loss, and the generated text.]*
- **Because:** *[Explain your reasoning based on what positional encodings provide to self-attention.]*

In [ ]:
# Run Experiment B2: Remove positional encoding
b2_config = GPTConfig(use_pos_encoding=False)
b2_model, results["B2"] = run_experiment(
    name="B2: No Positional Encoding",
    config=b2_config,
    generate_chars=500,
    measure_entropy=True,
)

# Compare loss curves on the same plot and axes
plot_loss_comparison(
    [results["Baseline"], results["B2"]],
    title="Experiment B2: Baseline vs. No Positional Encoding",
)

print("=" * 60)
print("B2 (NO POSITIONAL ENCODING) — 500-CHARACTER GENERATED SAMPLE")
print("=" * 60)
print(results["B2"]["sample"])
print("=" * 60)

#### B2 Report (Your Own Words)

- **Result (with numbers):** *[State the final validation loss and sample quality of B2 compared to Baseline in 1–2 sentences.]*
- **Explanation (and comparison to prediction):** *[Answer: How much does the model still "know" about order, and how is that possible with a causal mask? Explain where your prediction was right or wrong.]*

### B3: Removing the $\sqrt{d_k}$ Scaling in Attention

**Prediction (write before running B3 — do not edit afterwards):**

- **Experiment:** B3 — Remove the $1/\sqrt{d_k}$ scaling factor in attention (`use_scale = False`)
- **I predict:** *[Write what you expect to happen to train loss, validation loss, training stability, and Layer 1 attention entropy.]*
- **Because:** *[Explain your reasoning based on dot-product variance and softmax saturation.]*

In [ ]:
# Run Experiment B3: Remove 1/sqrt(d_k) scaling in attention
b3_config = GPTConfig(use_scale=False)
b3_model, results["B3"] = run_experiment(
    name="B3: No sqrt(d_k) Scaling",
    config=b3_config,
    generate_chars=500,
    measure_entropy=True,
)

# Side-by-side plots: (1) Loss comparison on same axes, (2) Layer 1 Attention Entropy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Loss curves
for run, color in [(results["Baseline"], "#1f77b4"), (results["B3"], "#d62728")]:
    steps = run["history"]["steps"]
    axes[0].plot(steps, run["history"]["train_loss"], linestyle="--", color=color, alpha=0.65,
                 label=f"{run['name']} — Train")
    axes[0].plot(steps, run["history"]["val_loss"], linestyle="-", marker="o", markersize=4,
                 linewidth=2, color=color, label=f"{run['name']} — Val")
axes[0].set_title("B3: Loss Curves (Baseline vs. No Scaling)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Training Steps")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Right plot: Layer 1 Average Attention Entropy (in nats)
for run, color in [(results["Baseline"], "#1f77b4"), (results["B3"], "#d62728")]:
    steps = run["history"]["steps"]
    axes[1].plot(steps, run["history"]["layer1_entropy"], linestyle="-", marker="s", markersize=4,
                 linewidth=2, color=color, label=run["name"])
axes[1].set_title("B3: Layer 1 Average Attention Entropy (nats)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Training Steps")
axes[1].set_ylabel("Mean Shannon Entropy (nats)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Baseline Layer 1 Avg Attention Entropy -> Start: {results['Baseline']['history']['layer1_entropy'][0]:.4f} | "
      f"Final: {results['Baseline']['final_layer1_entropy']:.4f}")
print(f"B3       Layer 1 Avg Attention Entropy -> Start: {results['B3']['history']['layer1_entropy'][0]:.4f} | "
      f"Final: {results['B3']['final_layer1_entropy']:.4f}")

#### B3 Report (Your Own Words)

- **Result (with numbers):** *[Report the final validation loss and Layer 1 average attention entropy for both Baseline and B3 in 1–2 sentences.]*
- **Explanation (and comparison to prediction):** *[Answer: Does softmax become peaky? Does training slow down or become unstable? Explain where your prediction was right or wrong.]*

### B4 (Bonus): Varying Number of Attention Heads (1 vs. 4 vs. 8)

*(Keep `d_model = 128` fixed so total parameter count remains identical.)*

**Prediction (write before running B4 — do not edit afterwards):**

- **Experiment:** B4 — Compare 1 head and 8 heads against the 4-head baseline (`d_model = 128`)
- **I predict:** *[Write what you expect to happen to validation loss for 1 head and 8 heads compared to 4 heads.]*
- **Because:** *[Explain your reasoning based on head dimension $d_k = d_{\text{model}} / h$ and multi-subspace representation.]*

In [ ]:
# Run Bonus Experiment B4: 1 head and 8 heads (keeping d_model = 128)
b4_1h_config = GPTConfig(n_head=1)
_, results["B4_1H"] = run_experiment(
    name="B4: 1 Head (d_k=128)",
    config=b4_1h_config,
    generate_chars=0,
    measure_entropy=False,
)

b4_8h_config = GPTConfig(n_head=8)
_, results["B4_8H"] = run_experiment(
    name="B4: 8 Heads (d_k=16)",
    config=b4_8h_config,
    generate_chars=0,
    measure_entropy=False,
)

# Plot 1 head, 4 heads (Baseline), and 8 heads on the same axes
plot_loss_comparison(
    [results["B4_1H"], results["Baseline"], results["B4_8H"]],
    title="Bonus B4: Effect of Number of Attention Heads (d_model = 128)",
)

#### B4 Report (Your Own Words)

- **Result (with numbers):** *[Compare final validation losses for 1 head, 4 heads, and 8 heads in 1–2 sentences.]*
- **Explanation (and comparison to prediction):** *[Answer: Do more heads help a model this small? Why or why not? Explain where your prediction was right or wrong.]*

### B5 (Bonus): Removing Residual Connections with 8 Layers

**Prediction (write before running B5 — do not edit afterwards):**

- **Experiment:** B5 — Compare an 8-layer Transformer with residuals vs. without residuals (`n_layer = 8`)
- **I predict:** *[Write what you expect to happen to the train and validation loss curves with and without residual connections.]*
- **Because:** *[Explain your reasoning based on gradient flow in deep networks.]*

In [ ]:
# Run Bonus Experiment B5: 8 layers with vs. without residual connections
b5_res_config = GPTConfig(n_layer=8, use_residual=True)
_, results["B5_8L_Res"] = run_experiment(
    name="B5: 8 Layers + Residuals",
    config=b5_res_config,
    generate_chars=0,
    measure_entropy=False,
)

b5_nores_config = GPTConfig(n_layer=8, use_residual=False)
_, results["B5_8L_NoRes"] = run_experiment(
    name="B5: 8 Layers, No Residuals",
    config=b5_nores_config,
    generate_chars=0,
    measure_entropy=False,
)

# Plot both 8-layer models (and Baseline for reference) on the same axes
plot_loss_comparison(
    [results["B5_8L_Res"], results["B5_8L_NoRes"], results["Baseline"]],
    title="Bonus B5: 8-Layer Transformer — With vs. Without Residual Connections",
)

#### B5 Report (Your Own Words)

- **Result (with numbers):** *[Compare final train and validation losses for the 8-layer network with and without residuals in 1–2 sentences.]*
- **Explanation (and comparison to prediction):** *[Answer: Why do deeper networks fail to train without residuals? Explain where your prediction was right or wrong.]*

In [ ]:
# Summary Table of All Completed Experiments + Save Results to JSON
print(f"{'Experiment':<28} | {'Params':>9} | {'Train Loss':>10} | {'Val Loss':>10} | {'L1 Entropy':>10} | {'Time (s)':>8}")
print("-" * 88)
for key, res in results.items():
    ent = f"{res['final_layer1_entropy']:.4f}" if res["final_layer1_entropy"] is not None else "N/A"
    print(f"{res['name']:<28} | {res['n_params']:>9,} | {res['final_train_loss']:>10.4f} | "
          f"{res['final_val_loss']:>10.4f} | {ent:>10} | {res['elapsed_sec']:>8.1f}")

# Save full numerical metrics so you can easily reference them while writing your analysis
with open("experiment_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("\nSaved all experiment metrics to experiment_results.json")

## 4. Discussion

> *[Write your overall reflection in your own words:]*
> - **Which component mattered most?** *(Cite numbers from your ablation table.)*
> - **Which component mattered least?** *(Cite numbers from your ablation table.)*
> - **What surprised you most?** *(Highlight any gap between your predictions and empirical results.)*

## 5. AI Usage Log

| Tool Used | Purpose / What It Helped With | Verified & Understood? |
| :--- | :--- | :--- |
| Gemini | Generated modular PyTorch `MiniGPT` scaffolding, ablation config flags, attention entropy calculation, and Matplotlib plotting code. | Yes — reviewed line-by-line |
| *[Add any other tool]* | *[e.g., Debugging / concept explanations]* | *[Yes/No]* |

*Note: All predictions, experimental analyses, and written discussions in this notebook were written independently in my own words in accordance with the lab AI tool policy.*